In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# plt.rcParams['font.family'] = 'Malgun Gothic' # For Windows
plt.rcParams["font.family"] = "AppleGothic"   # Mac
%matplotlib inline

In [2]:
import pandas as pd

df = pd.read_csv("./2025_Airbnb_NYC_listings.csv")
df.head()

,Unnamed: 0,id,source,name,description,neighborhood_overview,host_id,host_name,host_since,host_location,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,0,36121,city scrape,Lg Rm in Historic Prospect Heights,Cozy space share in the heart of a great neigh...,Full of tree-lined streets and beautiful brown...,62165,Michael,2009-12-11,"New York, NY",...,5.00,5.00,5.00,NaN,f,1,0,1,0,0.05
1,1,36647,city scrape,"1 Bedroom & your own Bathroom, Elevator Apartment",Private bedroom with your own bathroom in a 2 ...,"Manhattan, SE corner of 2nd Ave/ E. 110th street",157798,Irene,2010-07-04,"New York, NY",...,4.90,4.38,4.71,NaN,f,1,0,1,0,0.58
2,2,38663,city scrape,Luxury Brownstone in Boerum Hill,"Beautiful, large home in great hipster neighbo...","diverse, lively, hip, cool: loaded with restau...",165789,Sarah,2010-07-13,"New York, NY",...,4.88,4.86,4.62,OSE-STRREG-0001784,f,1,0,1,0,0.28
3,3,38833,city scrape,Spectacular West Harlem Garden Apt,This is a very large and unique space. An inc...,West Harlem is now packed with great restauran...,166532,Matthew,2010-07-14,"New York, NY",...,4.96,4.79,4.82,OSE-STRREG-0000476,f,1,1,0,0,1.36
4,4,39282,city scrape,“Work-from-home” from OUR home.,*Monthly Discount will automatically apply <br...,THE NEIGHBORHOOD:<br />Our apartment is locate...,168525,Gustavo,2010-07-16,"New York, NY",...,4.88,4.85,4.78,OSE-STRREG-0001150,f,2,0,2,0,1.54


### 데이트타입 변환

In [ ]:
# str 컬럼안에 date type이 숨어있다! datetime으로 변환
date_cols = ['host_since', 'first_review', 'last_review' , 'calendar_last_scraped']

df[date_cols] = df[date_cols].apply(pd.to_datetime,errors='coerce')

df[date_cols].dtypes

host_since               datetime64[us]
first_review             datetime64[us]
last_review              datetime64[us]
calendar_last_scraped    datetime64[us]
dtype: object

### price -> float 변환

In [4]:
# price 타입 str -> float 으로 변환
df['price']=(df['price'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False).astype(float))
df['price'].dtype

dtype('float64')

### price -> log 변환

In [5]:
# price 로그변환
df["price_log1p"] = np.log1p(df["price"])  # log(1+Fare)
df["price_log1p"]

0        5.303305
1        4.418841
2        6.641182
3        4.941642
4        4.875197
           ...   
22303    4.290459
22304    4.077537
22305    5.703782
22306    5.303305
22307    4.077537
Name: price_log1p, Length: 22308, dtype: float64

### vlaue_counts로 본 property_type의 종류를 거주용/숙박용 기준으로 나누어서 새 컬럼에 매핑.
- Residential   (주거형)
- Lodging       (숙박업형)

In [9]:
len(df['property_type'])

22308

In [6]:
property_map = {
    'Room in hotel': 'Lodging',
    'Room in boutique hotel': 'Lodging',

    'Entire rental unit': 'Residential',
    'Private room in rental unit': 'Residential',
    'Private room in home': 'Residential',
    'Entire home': 'Residential',
    'Entire condo': 'Residential',
    'Private room in townhouse': 'Residential',
    'Private room in condo': 'Residential',
    'Entire townhouse': 'Residential',
    'Entire loft': 'Residential',
    'Entire guest suite': 'Residential'
}

df['property_regulation_type'] = df['property_type'].map(property_map)

df['property_regulation_type'] = df['property_regulation_type'].fillna('Other')

In [7]:
df['property_regulation_type'].value_counts()

property_regulation_type
Residential    20335
Lodging         1026
Other            947
Name: count, dtype: int64

In [8]:
len(df['property_regulation_type'])

22308